# CS781 Phase 6 - Abstractive legal summarization with the Phase-1 models

One notebook for every Phase-6 experiment, built like the Phase-2 to Phase-5 notebooks. Set the experiment in **section 3** and run all, once per experiment.

| switch | options |
|---|---|
| `MODEL_TYPE` | `"gpt"`, `"modern"`, `"rnn"` (decoder-only: document + `Summary:` + summary in one sequence) or `"t5"` (encoder reads the document, decoder writes the summary) |
| `FT_METHOD` | `"full"`, `"partial"` (top layers), `"lora"`, `"lorma"`, `"pfeiffer"` (bottleneck adapter after every FFN), `"houlsby"` (adapters after attention **and** after FFN) |
| `USE_KD` | `True`: a smaller student (first `STUDENT_LAYERS` layers of `MODEL_TYPE`) learns from the saved teacher's token distributions (word-level KD) |
| `CONTEXT_MODE` | how a long judgment is fitted into the input: `"head"` (beginning), `"head_tail"` (beginning + end), `"select"` (most important sentences, Luhn-style word-frequency scores, kept in document order) |
| `DOMAIN_ADAPT_EPOCHS` | > 0: first continue language-model training on the Phase-6 documents, then train summarization |
| decoding | `DECODING` greedy / top-p, `NO_REPEAT_NGRAM`, `REPETITION_PENALTY`, `MIN_NEW_TOKENS`, `LENGTH_MODE` |

Pipeline: judgment -> sentences -> Phase-1 BPE -> context (`CONTEXT_MODE`, fits the model's input) -> Phase-1 model + adaptation -> generation with KV cache (Phase-1 loop) -> summary -> `prediction.json` (list format).

Local scores are on a held-out validation split of `train.json` (the brief gives no dev set): ROUGE-1/2/L (own code) and BERTScore (`bert_score` package, evaluation only, as in Phase 1).

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import gc
import torch

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE = "Phase_6_Summ/Test/test.json"

# ---------------- the experiment ----------------
EXPERIMENT_NAME = "modern_lorma_select"    # <-- CHANGED for this run
MODEL_TYPE = "modern"    # <-- CHANGED for this run
FT_METHOD = "lorma"    # <-- CHANGED for this run
SMOKE_TEST = False
USE_PHASE1_WEIGHTS = True    # False = random initialisation (ablation: are the Phase-1 representations useful?)
DOMAIN_ADAPT_EPOCHS = 0      # > 0: continue LM training on the Phase-6 documents first (gpt / modern / rnn)

# long documents
CONTEXT_MODE = "select"    # <-- CHANGED for this run
TOTAL_TOKENS = 2048          # decoder-only: <s> + document + "Summary:" + summary + </s> (Phase-1 context = 2048)
RNN_TOTAL_TOKENS = 1536      # the LSTM runs token by token, so it gets a shorter sequence (time)
T5_DOC_TOKENS = 1536         # T5 encoder input (Phase 1 trained it on 1024; relative positions allow more)
SUMMARY_TOKENS = "auto"      # longest summary in tokens (training + generation); "auto" = 90th percentile of train summaries, at most 1024
VAL_DOCS = 300               # held out from train.json for model selection (loss) and local scores
VAL_GEN_DOCS = 100           # of those, how many get a generated summary (ROUGE / BERTScore / analysis)
MAX_TRAIN_DOCS = None        # subset of the training documents (None = all)

# generation
DECODING = "greedy"          # "greedy" | "top_p"
TEMPERATURE = 0.7
TOP_P = 0.9
NO_REPEAT_NGRAM = 3          # never repeat a token 3-gram (redundancy control); 0 = off
REPETITION_PENALTY = 1.2     # > 1 makes already used tokens less likely; 1.0 = off
MIN_NEW_TOKENS = "auto"      # </s> is blocked before this many tokens; "auto" = half the 10th percentile of train summaries
LENGTH_MODE = "tuned"        # "generated" (as the model stops) | "tuned" (cut to the word length that scores best on validation)
GEN_BATCH_SIZE = 8
LOCAL_BERTSCORE = True       # BERTScore on the validation summaries (downloads the scorer model; evaluation only)

# knowledge distillation (Section 10.2.2): student = first STUDENT_LAYERS layers of MODEL_TYPE
USE_KD = False
STUDENT_LAYERS = 2
TEACHER_MODEL_TYPE = "modern"
TEACHER_FT_METHOD = "full"
TEACHER_SUMM_PATH = "/kaggle/input/datasets/medinim/phase6-teacher/summ_model.pt"
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5               # loss = KD_ALPHA * CE(reference) + (1 - KD_ALPHA) * T^2 * KL(teacher || student), on summary tokens
# (the teacher must be decoder-only when the student is decoder-only; it reads the student's input sequence)

# LoRA / LoRMA / adapters / partial
ADAPTER_RANK = 16
ADAPTER_ALPHA = 32
BOTTLENECK = 64              # "houlsby" / "pfeiffer": size of the adapter bottleneck
PREFIX_LENGTH = 64           # "prefix": number of trainable key/value positions per layer (gpt / modern)
PREFIX_HIDDEN = 512          # "prefix": hidden size of the reparametrization MLP (Li & Liang 2021, Module 4.1)
T5_TASK_PREFIX = "summarize:"   # T5 input = "summarize: <document>" (text-to-text format, Module 2.4)
PARTIAL_TOP_LAYERS = 2       # "partial": only the top layers (+ final norm) are trained

# Phase-1 models: one Kaggle dataset with model_<type>.safetensors and bpe_legal_<type>.json
PHASE1_FOLDER = "/kaggle/input/datasets/medinim/phase1-models"   # searched automatically below if this path is wrong
if not os.path.exists(os.path.join(PHASE1_FOLDER, "model_modern.safetensors")):
  import glob
  hits = glob.glob("/kaggle/input/**/model_modern.safetensors", recursive=True)
  if hits:
    PHASE1_FOLDER = os.path.dirname(hits[0])
print("Phase-1 models from:", PHASE1_FOLDER)
PHASE1_MODEL_FILES = {k: os.path.join(PHASE1_FOLDER, f"model_{k}.safetensors") for k in ["gpt", "modern", "rnn", "t5"]}
PHASE1_BPE_FILES = {k: os.path.join(PHASE1_FOLDER, f"bpe_legal_{k}.json") for k in ["gpt", "modern", "rnn", "t5"]}

# Phase-1 architecture sizes: must be EXACTLY what Phase 1 trained with
MODEL_CONTEXT_LENGTH = 2048
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 640, 10, 10, 5
D_FF = 4 * N_EMBD
RNN_CELL, RNN_HIDDEN, RNN_LAYERS = "lstm", 1152, 4
T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS, T5_HEADS, T5_D_FF, T5_N_SENTINELS = 512, 8, 8, 8, 2048, 100
DROPOUT = 0.1
LOAD_SUMM_PATH = None        # path to a saved summ_model.pt -> skip training, only generate ("auto" = find <LOAD_FROM_EXPERIMENT>/summ_model.pt in the inputs)
LOAD_FROM_EXPERIMENT = None  # with "auto": which trained experiment to reload (None = EXPERIMENT_NAME); used for decoding-only runs
REQUIRE_GPU = True

# training (Trainer, like the earlier phases)
if FT_METHOD in ("full", "partial"):
  LEARNING_RATE = 1e-4
else:
  LEARNING_RATE = 5e-4       # LoRA / LoRMA / adapters
if MODEL_TYPE == "rnn":
  LEARNING_RATE = 5e-4       # Phase 5: the LSTM did not move at 5e-5
if MODEL_TYPE == "t5":
  LEARNING_RATE = 1e-3       # T5 needs a larger learning rate (Phases 1, 2, 5)
LR_OVERRIDE = None           # set a number to force the learning rate
if LR_OVERRIDE:
  LEARNING_RATE = LR_OVERRIDE
EPOCHS = 3
BATCH_SIZE = 1               # sequences per GPU (2048 tokens each)
GRAD_ACCUM = 8               # 2 GPUs x 1 x 8 = 16 documents per update
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if SMOKE_TEST:
  EPOCHS = 1
  MAX_TRAIN_DOCS = 64
  VAL_DOCS = 16
  VAL_GEN_DOCS = 8
if REQUIRE_GPU and not torch.cuda.is_available():
  raise RuntimeError("No GPU! In Kaggle: Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
if USE_KD and not os.path.exists(TEACHER_SUMM_PATH):
  import glob
  hits = glob.glob("/kaggle/input/**/summ_model.pt", recursive=True)
  if hits:
    TEACHER_SUMM_PATH = hits[0]
  print("KD teacher:", TEACHER_SUMM_PATH)
if LOAD_SUMM_PATH == "auto":
  import glob
  src = LOAD_FROM_EXPERIMENT or EXPERIMENT_NAME
  hits = glob.glob(f"/kaggle/input/**/{src}/summ_model.pt", recursive=True)
  if not hits:
    raise FileNotFoundError(f"no {src}/summ_model.pt found: Add Input -> the output of the notebook that trained {src}")
  LOAD_SUMM_PATH = hits[0]
  print("generate only, using", LOAD_SUMM_PATH)

OUTPUT_DIR = "/kaggle/working/cs781_phase6"
EXP_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
os.makedirs(EXP_DIR, exist_ok=True)
PREDICTION_PATH = os.path.join(EXP_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, f"submission_{EXPERIMENT_NAME}.zip")
RESULTS_PATH = os.path.join(OUTPUT_DIR, "results.jsonl")
print("Experiment:", EXPERIMENT_NAME, "|", MODEL_TYPE, FT_METHOD, "|", CONTEXT_MODE, "| KD" if USE_KD else "")

## 4. Tokenizer (the Phase-1 BPE, unchanged)

In [ ]:
import json
import heapq
from collections import Counter, defaultdict

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
WORD_END = "</w>"

class ByteLevelBPE:
  def __init__(self, vocab_size):
      self.vocab_size = vocab_size
      self.merges = []
      self.merge_ranks = {}
      self.itos = []
      self.stoi = {}
      self._cache = {}
      self.pad_id = None
      self.bos_id = None
      self.eos_id = None

  def _word_to_symbols(self, word):
      byte_vals = word.encode("utf-8")
      symbols = [chr(b) for b in byte_vals]
      symbols.append(WORD_END)
      return symbols

  def train(self, sentences):
      word_freq = Counter()
      for sent in sentences:
          for word in sent.split():
              word_freq[word] += 1

      words = list(word_freq)
      freqs = [word_freq[w] for w in words]
      word_symbols = [self._word_to_symbols(w) for w in words]

      vocab_list = list(SPECIALS)
      vocab_seen = set(vocab_list)

      initial_symbols = {chr(b) for b in range(256)} | {WORD_END}

      for s in sorted(initial_symbols):
          if s not in vocab_seen:
              vocab_list.append(s)
              vocab_seen.add(s)

      # this loop recounts every pair every merge, which is far too slow
      # for the legal corpus. Replace it with incremental pair counts + pair_to_words.
      pair_counts = Counter()
      pair_to_words = defaultdict(set)
      for wi, symbols in enumerate(word_symbols):
          f = freqs[wi]
          for i in range(len(symbols) - 1):
              p = (symbols[i], symbols[i + 1])
              pair_counts[p] += f
              pair_to_words[p].add(wi)

      heap = [(-c, p) for p, c in pair_counts.items()]
      heapq.heapify(heap)

      while len(vocab_list) < self.vocab_size:
          best_pair, best_count = None, 0
          while heap:
              negc, p = heapq.heappop(heap)
              if pair_counts.get(p, 0) == -negc and -negc > 0:
                  best_pair, best_count = p, -negc
                  break
          if best_pair is None or best_count <= 1:
              break

          a, b = best_pair
          merged = a + b
          changed = set()

          for wi in list(pair_to_words[best_pair]):
              symbols = word_symbols[wi]
              f = freqs[wi]

              old_pairs = Counter(zip(symbols, symbols[1:]))   # positions, so overlaps count
              new_symbols = []
              i = 0
              while i < len(symbols):
                  if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                      new_symbols.append(merged)
                      i += 2
                  else:
                      new_symbols.append(symbols[i])
                      i += 1
              word_symbols[wi] = new_symbols
              new_pairs = Counter(zip(new_symbols, new_symbols[1:]))

              for p, c in old_pairs.items():
                  pair_counts[p] -= c * f
                  changed.add(p)
                  if p not in new_pairs:
                      pair_to_words[p].discard(wi)
              for p, c in new_pairs.items():
                  pair_counts[p] += c * f
                  changed.add(p)
                  pair_to_words[p].add(wi)

          for p in changed:
              c = pair_counts[p]
              if c > 0:
                  heapq.heappush(heap, (-c, p))
              else:
                  del pair_counts[p]
                  pair_to_words.pop(p, None)

          self.merges.append([a, b])
          self.merge_ranks[(a, b)] = len(self.merges) - 1

          if merged not in vocab_seen:
              vocab_list.append(merged)
              vocab_seen.add(merged)

      self.itos = vocab_list
      self.stoi = {tok: i for i, tok in enumerate(self.itos)}
      self.pad_id = self.stoi["<pad>"]
      self.bos_id = self.stoi["<s>"]
      self.eos_id = self.stoi["</s>"]
      return self

  def encode_word(self, word):
      if word in self._cache:
        return self._cache[word]
      symbols = self._word_to_symbols(word)
      while len(symbols) > 1:
          pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
          ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
          if not ranked:
              break
          _, i = min(ranked)
          symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

      self._cache[word] = symbols
      return symbols

  def encode_ids(self, sentence):
      unk = self.stoi["<unk>"]
      return [self.stoi.get(t, unk) for t in self.encode(sentence)]

  def decode(self, ids):
      text = "".join(self.itos[i] for i in ids if self.itos[i] not in SPECIALS)
      words = text.split(WORD_END)
      out = [bytes(ord(c) for c in w).decode("utf-8", errors="replace") for w in words if w]
      return " ".join(out)

  def encode(self, sentence):
      tokens = []
      for word in sentence.split():
          tokens.extend(self.encode_word(word))
      return tokens

  def save(self, path):
      with open(path, "w", encoding="utf-8") as f:
          json.dump({"merges": self.merges, "itos": self.itos,
                      "vocab_size": self.vocab_size}, f, ensure_ascii=False)

  @staticmethod
  def load_bpe(path):
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      bpe = ByteLevelBPE(vocab_size=data["vocab_size"])
      bpe.merges = data["merges"]
      bpe.itos = data["itos"]
      bpe.stoi = {tok: i for i, tok in enumerate(bpe.itos)}
      bpe.merge_ranks = {(a, b): i for i, (a, b) in enumerate(bpe.merges)}
      bpe.pad_id = bpe.stoi["<pad>"]
      bpe.bos_id = bpe.stoi["<s>"]
      bpe.eos_id = bpe.stoi["</s>"]
      return bpe

In [ ]:
import hashlib
for k, f in PHASE1_BPE_FILES.items():   # all four must be the same tokenizer (needed for KD)
  print(k, hashlib.md5(open(f, "rb").read()).hexdigest())
bpe = ByteLevelBPE.load_bpe(PHASE1_BPE_FILES[MODEL_TYPE])
UNK_ID = bpe.stoi["<unk>"]
print("vocab size:", len(bpe.itos))

## 5. Download the Phase-6 data, hold out a validation split

`train.json` has `{"id", "document", "summary"}`; `test.json` has `{"id", "document"}`. There is no dev set, so `VAL_DOCS` training documents are held out (fixed seed) for model selection and local scores.

Every document is split into sentences, and every sentence is tokenized once. The input context of a document is then built from sentences (section 6).

In [ ]:
from huggingface_hub import hf_hub_download
from collections import Counter
import re
import numpy as np

def load_json_file(fname):
  p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
  with open(p, "r", encoding="utf-8") as f:
    return json.load(f)

def as_text(x):
  if isinstance(x, list):
    return " ".join(str(t) for t in x)
  return str(x) if x is not None else ""

def read_docs(data, with_summary):
  if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
    data = data["data"]
  if isinstance(data, dict):                     # {"doc_001": {...}} or {"doc_001": "text"}
    items = []
    for k, v in data.items():
      v = {"document": v} if isinstance(v, str) else dict(v)
      v.setdefault("id", k)
      items.append(v)
    data = items
  docs = []
  for i, d in enumerate(data):
    text = as_text(d.get("document", d.get("text", d.get("judgment", ""))))
    summ = None
    if with_summary:
      for key in ("summary", "abstract", "headnote", "target"):
        if key in d:
          summ = as_text(d[key])
          break
      if summ is None:
        raise ValueError(f"no summary in example {d.get('id', i)}; keys: {list(d.keys())}")
    docs.append({"id": d.get("id", i), "document": text, "summary": summ})
  return docs

all_train = read_docs(load_json_file(TRAIN_FILE), True)
test_docs = read_docs(load_json_file(TEST_FILE), False)
rng = random.Random(SEED)
order = list(range(len(all_train)))
rng.shuffle(order)
val_docs = [all_train[i] for i in order[:VAL_DOCS]]
train_docs = [all_train[i] for i in order[VAL_DOCS:]]
if MAX_TRAIN_DOCS is not None:
  train_docs = train_docs[:MAX_TRAIN_DOCS]
print(f"train.json: {len(all_train)} documents -> train {len(train_docs)} | validation {len(val_docs)} | test {len(test_docs)}")

def pct(values, ps=(10, 50, 90)):
  v = sorted(values)
  return {p: v[min(len(v) - 1, int(p / 100 * len(v)))] for p in ps}

dw = [len(d["document"].split()) for d in all_train]
sw = [len(d["summary"].split()) for d in all_train]
tw = [len(d["document"].split()) for d in test_docs]
print("words per document (10/50/90 %):", pct(dw), "| max", max(dw))
print("words per summary  (10/50/90 %):", pct(sw), "| max", max(sw))
print("words per test document (10/50/90 %):", pct(tw))
ratios = [s / max(d, 1) for s, d in zip(sw, dw)]
print("summary / document length (10/50/90 %):", {k: round(v, 3) for k, v in pct(ratios).items()})

def split_sentences(text):
  # simple rule: a sentence ends at . ? ! or ; followed by a space and a capital letter / digit / bracket
  text = re.sub(r"\s+", " ", text).strip()
  parts = re.split(r"(?<=[.?!;])\s+(?=[A-Z0-9(\[\"'])", text)
  out = []
  for p in parts:
    w = p.split()
    while len(w) > 80:                     # very long "sentences" (lists, tables) are cut every 80 words
      out.append(" ".join(w[:80]))
      w = w[80:]
    if w:
      out.append(" ".join(w))
  return out

t0 = time.time()
for d in train_docs + val_docs + test_docs:
  d["sents"] = split_sentences(d["document"])
  d["sent_ids"] = [bpe.encode_ids(s) for s in d["sents"]]
  d["n_tokens"] = sum(len(s) for s in d["sent_ids"])
  if d["summary"] is not None:
    d["summ_ids"] = bpe.encode_ids(d["summary"])
bpe._cache.clear()
print(f"tokenized {len(train_docs) + len(val_docs) + len(test_docs)} documents in {time.time() - t0:.0f}s")
st = [len(d["summ_ids"]) for d in train_docs]
print("tokens per document (10/50/90 %):", pct([d["n_tokens"] for d in train_docs]))
print("tokens per summary  (10/50/90 %):", pct(st))
print(f"BPE tokens per word: documents {sum(d['n_tokens'] for d in train_docs) / max(1, sum(len(d['document'].split()) for d in train_docs)):.2f}")

if SUMMARY_TOKENS == "auto":
  SUMMARY_TOKENS = int(min(1024, pct(st, (90,))[90]))
if MIN_NEW_TOKENS == "auto":
  MIN_NEW_TOKENS = int(pct(st, (10,))[10] // 2)
SEP_IDS = bpe.encode_ids("Summary:")
if MODEL_TYPE == "t5":
  T5_PREFIX_IDS = bpe.encode_ids(T5_TASK_PREFIX) if T5_TASK_PREFIX else []
  DOC_BUDGET = T5_DOC_TOKENS - len(T5_PREFIX_IDS)
else:
  total = RNN_TOTAL_TOKENS if MODEL_TYPE == "rnn" else TOTAL_TOKENS
  if FT_METHOD == "prefix":
    total -= PREFIX_LENGTH                    # the prefix uses positions of the 2048-token attention window
  DOC_BUDGET = total - SUMMARY_TOKENS - len(SEP_IDS) - 2
print(f"SUMMARY_TOKENS = {SUMMARY_TOKENS} | MIN_NEW_TOKENS = {MIN_NEW_TOKENS} | document budget = {DOC_BUDGET} tokens "
      f"({100 * np.mean([min(1, DOC_BUDGET / max(1, d['n_tokens'])) for d in train_docs]):.0f}% of an average document fits)")
assert DOC_BUDGET >= 256, "the summary is too long for the context: lower SUMMARY_TOKENS"

## 6. Long documents -> the model's input context

A judgment is usually much longer than `DOC_BUDGET` tokens, so only part of it can be read:

* `"head"`: the first `DOC_BUDGET` tokens.
* `"head_tail"`: the first half of the budget from the beginning (parties, facts) and the second half from the end (reasoning, decision).
* `"select"` (importance-based selection): every sentence gets a score = average document frequency of its content words (Luhn 1958, the classic frequency-based extractive summarizer). The first 3 sentences are always kept; then the best sentences are added until the budget is full, and they are put back in document order.

**Coverage check (no model needed):** for each mode, the share of the reference summary's content words that are still present in the input. This shows how much information each strategy throws away before the model even starts.

In [ ]:
STOPWORDS = set("""the a an and or of to in on at by for with from as is was were be been are that this these those it its
which who whom whose not no but if then than so such has have had do does did can could may might shall should will would
said also there their they them he she his her him i we our you your any all said upon into under over between per""".split())

def content_words(text):
  return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2 and w not in STOPWORDS]

def flatten(list_of_lists):
  out = []
  for x in list_of_lists:
    out.extend(x)
  return out

def sentence_scores(d):
  freq = Counter(content_words(d["document"]))
  scores = []
  for s in d["sents"]:
    w = content_words(s)
    scores.append(sum(freq[x] for x in set(w)) / (len(w) + 5) if w else 0.0)
  return scores

def build_context(d, mode, budget):
  ids = flatten(d["sent_ids"])
  if len(ids) <= budget:
    return ids
  if mode == "head":
    return ids[:budget]
  if mode == "head_tail":
    h = budget // 2
    return ids[:h] + ids[len(ids) - (budget - h):]
  if mode == "select":
    scores = sentence_scores(d)
    keep = set(range(min(3, len(d["sents"]))))
    used = sum(len(d["sent_ids"][i]) for i in keep)
    for i in sorted(range(len(scores)), key=lambda j: -scores[j]):
      if i in keep:
        continue
      n = len(d["sent_ids"][i])
      if used + n > budget:
        continue
      keep.add(i)
      used += n
    return flatten(d["sent_ids"][i] for i in sorted(keep))[:budget]
  raise ValueError(mode)

def coverage(context_ids, summary):
  ref = set(content_words(summary))
  have = set(content_words(bpe.decode(context_ids)))
  return len(ref & have) / max(1, len(ref))

print(f"coverage of the reference summary's content words by the input ({len(val_docs)} validation documents, budget {DOC_BUDGET} tokens):")
cov = {}
for mode in ["head", "head_tail", "select"]:
  cov[mode] = float(np.mean([coverage(build_context(d, mode, DOC_BUDGET), d["summary"]) for d in val_docs]))
cov["whole document"] = float(np.mean([coverage(flatten(d["sent_ids"]), d["summary"]) for d in val_docs]))
for k, v in cov.items():
  print(f"   {k:15s} {100 * v:.1f}%" + ("   <- used" if k == CONTEXT_MODE else ""))

## 7. Training sequences

* **Decoder-only** (`gpt`, `modern`, `rnn`): `<s> context Summary: summary </s>` in one sequence. The loss is only on the summary tokens (labels = -100 elsewhere), so the model learns to write the summary, not to copy the judgment.
* **T5**: the encoder reads the context, the decoder gets `<s> summary` and predicts `summary </s>`.
* `DOMAIN_ADAPT_EPOCHS > 0`: before that, plain language-model training on blocks of the Phase-6 documents (labels = inputs).

In [ ]:
from datasets import Dataset

def make_example(d):
  ctx = build_context(d, CONTEXT_MODE, DOC_BUDGET)
  summ = d["summ_ids"][:SUMMARY_TOKENS] + [bpe.eos_id]
  if MODEL_TYPE == "t5":
    return {"input_ids": T5_PREFIX_IDS + ctx, "decoder_input_ids": [bpe.bos_id] + summ[:-1], "labels": summ}
  prompt = [bpe.bos_id] + ctx + SEP_IDS
  return {"input_ids": prompt + summ, "labels": [-100] * len(prompt) + summ}

def to_dataset(docs):
  rows = [make_example(d) for d in docs]
  return Dataset.from_dict({k: [r[k] for r in rows] for k in rows[0]})

train_dataset = to_dataset(train_docs)
val_dataset = to_dataset(val_docs)
print(train_dataset)
n = [len(x) for x in train_dataset["input_ids"]]
print(f"training sequence length: mean {np.mean(n):.0f}, max {max(n)}")

def pad_to(seqs, value):
  L = max(len(s) for s in seqs)
  return torch.tensor([s + [value] * (L - len(s)) for s in seqs], dtype=torch.long)

def data_collator(features):
  # right padding: causal models never look at later (padding) positions, the loss ignores -100
  batch = {"input_ids": pad_to([f["input_ids"] for f in features], bpe.pad_id),
           "labels": pad_to([f["labels"] for f in features], -100)}
  if "decoder_input_ids" in features[0]:
    batch["decoder_input_ids"] = pad_to([f["decoder_input_ids"] for f in features], bpe.pad_id)
  return batch

dapt_dataset = None
if DOMAIN_ADAPT_EPOCHS > 0 and MODEL_TYPE != "t5":
  L = RNN_TOTAL_TOKENS if MODEL_TYPE == "rnn" else TOTAL_TOKENS
  blocks = []
  for d in train_docs:
    ids = [bpe.bos_id] + flatten(d["sent_ids"])
    for s in range(0, min(len(ids), 4 * L), L):          # at most 4 blocks per document
      if len(ids[s:s + L]) > 64:
        blocks.append(ids[s:s + L])
  dapt_dataset = Dataset.from_dict({"input_ids": blocks, "labels": blocks})
  print(f"domain adaptation: {len(blocks)} blocks of up to {L} tokens from the training documents")

## 8. Phase-1 model code (copied unchanged from the Phase-1 notebook)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Module 2.3:  RMS(x) = sqrt( (1/E) * sum_i x_i^2 + eps ),   RMSNorm(x)_i = gamma_i * x_i / RMS(x)"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable, no bias (beta)

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)


# ---- RoPE (Module 2.3 / 3.1, Assignment 3 "rope.py") ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    # theta_i = base^(-2(i-1)/d) for every pair of dimensions; angle = position m * theta_i
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # split the last dim in half and return (-x2, x1)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos, sin: (B, 1, T, d_head)
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention (Module 2.3, Assignment 3).
    n_kv_heads == n_heads -> normal MHA,  n_kv_heads < n_heads -> grouped-query attention (GQA).
    past_kv = (k, v) of earlier tokens -> KV cache for generation (Module 2.3).
    attention_mask (B, S): 1 = real token, 0 = padding (Module 2.3: masks prevent attention over padding)."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_ids].unsqueeze(1)   # (B, 1, T, d_head)
            sin = self.rope_sin[position_ids].unsqueeze(1)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_kv is not None:                  # KV cache: reuse keys/values of earlier tokens
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v)

        if self.n_rep > 1:                       # GQA: each k/v head is shared by n_rep query heads
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)

        S = k.size(2)                             # number of keys = cached + new
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale        # (B, H, T, S)
        allowed = self.causal_mask[S - T:S, :S].unsqueeze(0).unsqueeze(0)  # (1, 1, T, S)
        if attention_mask is not None:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            # a left-padding position has no real token before it, so let every
            # position always see itself (otherwise its whole row would be masked)
            itself = torch.eye(S, dtype=torch.bool, device=x.device)[S - T:S, :S]
            allowed = allowed | itself.unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~allowed, float("-inf"))
        att = F.softmax(scores, dim=-1)
        att = self.attn_dropout(att)
        out = att @ v                                                      # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out, new_kv


class FeedForward(nn.Module):
    """Assignment 3: two linear layers with a GELU in between, plus dropout."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class SwiGLUFeedForward(nn.Module):
    """Module 3.1 names SwiGLU as the modern feed-forward activation:
    FFN(x) = W2( SiLU(W1 x) * (W3 x) ),  SiLU(a) = a * sigmoid(a)"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w3 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a = self.w1(x)
        return self.dropout(self.w2(a * torch.sigmoid(a) * self.w3(x)))

In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 10
    n_heads: int = 10
    n_kv_heads: int = 10            # == n_heads for plain MHA
    d_ff: int = 2560
    max_seq_len: int = 2048
    dropout: float = 0.1
    pos_emb_type: str = "learned"   # "learned" (GPT-2 / BERT, Module 2.3) or "rope"
    norm_type: str = "layernorm"    # "layernorm" or "rmsnorm"
    ffn_type: str = "gelu"          # "gelu" or "swiglu"


class Block(nn.Module):
    """Pre-norm block (Module 2.3, used by GPT-2 and modern LLMs):  x + Attn(Norm(x)),  then  x + FFN(Norm(x))"""
    def __init__(self, cfg):
        super().__init__()
        if cfg.norm_type == "layernorm":
            self.ln1 = nn.LayerNorm(cfg.d_model)
            self.ln2 = nn.LayerNorm(cfg.d_model)
        else:
            self.ln1 = RMSNorm(cfg.d_model)
            self.ln2 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"))
        if cfg.ffn_type == "gelu":
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        else:
            self.ffn = SwiGLUFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        a, new_kv = self.attn(self.ln1(x), position_ids, attention_mask, past_kv)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, new_kv


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)    # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        if cfg.norm_type == "layernorm":
            self.ln_f = nn.LayerNorm(cfg.d_model)
        else:
            self.ln_f = RMSNorm(cfg.d_model)
        # Module 2.1 "Proper Weight Initialization": Xavier/Glorot for the embedding matrices.
        # PyTorch's default N(0, 1) embedding + weight tying gives huge first logits (start loss ~80)
        nn.init.xavier_uniform_(self.tok_emb.weight)
        if cfg.pos_emb_type == "learned":
            nn.init.xavier_uniform_(self.pos_emb.weight)

    def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.tok_emb(input_ids)
        if self.cfg.pos_emb_type == "learned":
            x = x + self.pos_emb(position_ids)
        x = self.drop(x)

        new_past = []
        for i, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, position_ids, attention_mask, layer_past)
            new_past.append(kv)
        x = self.ln_f(x)

        if use_cache:
            x = x[:, -1:, :]              # generation only needs the prediction for the last position
        logits = x @ self.tok_emb.weight.t()   # weight tying

        loss = None
        if labels is not None:
            # predict token t+1 from positions <= t (labels are shifted inside, like GPT-2)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_past}
        return {"loss": loss, "logits": logits}

In [ ]:
class LSTMCell(nn.Module):
    """One time step of an LSTM (Assignment 2): packed linear layers compute f, i, g, o at once."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 4 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)
        with torch.no_grad():
            self.x2g.bias[0:hidden_size] = 1.0   # forget gate bias close to 1 (Module 2.1)
            self.h2g.bias[0:hidden_size] = 0.0

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.x2g(x_t) + self.h2g(h_prev)
        f_pre, i_pre, g_pre, o_pre = gates.chunk(4, dim=-1)
        f_t = torch.sigmoid(f_pre)
        i_t = torch.sigmoid(i_pre)
        g_t = torch.tanh(g_pre)
        o_t = torch.sigmoid(o_pre)
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return (h_t, c_t)


class GRUCell(nn.Module):
    """One time step of a GRU with the Module 2.1 equations (layer names as in Assignment 2)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2z = nn.Linear(input_size, hidden_size)
        self.h2z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2r = nn.Linear(input_size, hidden_size)
        self.h2r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2n = nn.Linear(input_size, hidden_size)
        self.h2n = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        z_t = torch.sigmoid(self.x2z(x_t) + self.h2z(h_prev))          # update gate
        r_t = torch.sigmoid(self.x2r(x_t) + self.h2r(h_prev))          # reset gate
        n_t = torch.tanh(self.x2n(x_t) + self.h2n(r_t * h_prev))       # candidate h~
        h_t = (1 - z_t) * h_prev + z_t * n_t
        return h_t


class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.is_lstm = (cell_type == "lstm")
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        cell_cls = LSTMCell if self.is_lstm else GRUCell
        self.cells = nn.ModuleList([cell_cls(hidden_size, hidden_size) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.tok_emb.weight)     # Module 2.1: Xavier/Glorot initialisation

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None, use_cache=False):
        """past_key_values here = list of per-layer states (h, c) / h, carried during generation.
        attention_mask (B, T): at padding positions the state is NOT updated."""
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        new_states = []
        for l in range(self.num_layers):
            cell = self.cells[l]
            if past_key_values is not None:
                state = past_key_values[l]
            else:
                h = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
                state = (h, torch.zeros_like(h)) if self.is_lstm else h
            outputs = []
            for t in range(T):           # explicit unroll over time (BPTT through these steps)
                new_state = cell(x[:, t, :], state)
                if attention_mask is not None:
                    m = attention_mask[:, t].unsqueeze(1).to(x.dtype)     # 1 = real token, 0 = pad
                    if self.is_lstm:
                        new_state = (m * new_state[0] + (1 - m) * state[0], m * new_state[1] + (1 - m) * state[1])
                    else:
                        new_state = m * new_state + (1 - m) * state
                state = new_state
                outputs.append(state[0] if self.is_lstm else state)
            new_states.append(state)
            x = self.dropout(torch.stack(outputs, dim=1))    # input of the next layer

        if use_cache:
            x = x[:, -1:, :]
        logits = x @ self.tok_emb.weight.t()       # weight tying (Assignment 3)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_states}
        return {"loss": loss, "logits": logits}

In [ ]:
class T5RelativePositionBias(nn.Module):
    def __init__(self, n_heads, bidirectional, num_buckets=32, max_distance=128):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.rel_emb = nn.Embedding(num_buckets, n_heads)     # one scalar per (bucket, head)

    def bucket(self, relative_position):
        # relative_position = key_pos - query_pos
        n = -relative_position
        num_buckets = self.num_buckets
        ret = torch.zeros_like(n)
        if self.bidirectional:                 # encoder: half the buckets for keys on the right
            num_buckets = num_buckets // 2
            ret = ret + (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:                                  # decoder: only keys on the left exist
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2           # small offsets get their own bucket
        is_small = n < max_exact
        # bigger offsets: logarithmically wider buckets, everything beyond max_distance shares the last one
        large = max_exact + (torch.log(torch.clamp(n, min=1).float() / max_exact)
                             / math.log(self.max_distance / max_exact)
                             * (num_buckets - max_exact)).long()
        large = torch.clamp(large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        buckets = self.bucket(k_pos - q_pos)               # (q, k)
        bias = self.rel_emb(buckets)                       # (q, k, H)
        return bias.permute(2, 0, 1).unsqueeze(0)          # (1, H, q, k)


class T5Attention(nn.Module):
    """Self-attention (x_kv = x_q) or cross-attention (x_kv = encoder output)."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, position_bias=None, allowed=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.size(1)
        q = self.q_proj(x_q).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if position_bias is not None:
            scores = scores + position_bias
        if allowed is not None:
            scores = scores.masked_fill(~allowed, float("-inf"))
        att = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class T5FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))   # dropout inside the FFN


class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, position_bias, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, src_allowed))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = T5Attention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, position_bias, causal_allowed, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, causal_allowed))   # masked self-attention
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, None, src_allowed))        # cross-attention
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


class T5Model(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.shared_emb = nn.Embedding(vocab_size, d_model)    # encoder input, decoder input and output
        self.dropout = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([T5EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.dec_layers = nn.ModuleList([T5DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        self.enc_pos_bias = T5RelativePositionBias(n_heads, bidirectional=True)
        self.dec_pos_bias = T5RelativePositionBias(n_heads, bidirectional=False)
        nn.init.xavier_uniform_(self.shared_emb.weight)  # Module 2.1: Xavier/Glorot initialisation

    def encode(self, input_ids):
        S = input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]          # no attention to padding
        position_bias = self.enc_pos_bias(S, S, input_ids.device)
        x = self.dropout(self.shared_emb(input_ids))
        for layer in self.enc_layers:
            x = layer(x, position_bias, src_allowed)
        return self.dropout(self.enc_norm(x))

    def decode(self, decoder_input_ids, enc_out, input_ids, last_only=False):
        T = decoder_input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]
        causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool, device=enc_out.device))[None, None]
        position_bias = self.dec_pos_bias(T, T, enc_out.device)
        x = self.dropout(self.shared_emb(decoder_input_ids))
        for layer in self.dec_layers:
            x = layer(x, enc_out, position_bias, causal_allowed, src_allowed)
        x = self.dropout(self.dec_norm(x))
        if last_only:
            x = x[:, -1:, :]
        return x @ self.shared_emb.weight.t()                                # weight sharing

    def forward(self, input_ids, decoder_input_ids, labels=None):
        enc_out = self.encode(input_ids)
        logits = self.decode(decoder_input_ids, enc_out, input_ids)
        loss = None
        if labels is not None:
            # decoder_input_ids are already shifted (<s> + target[:-1]); -100 = padding, ignored
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}

### 8.5 Load a Phase-1 model and add the adaptation method

* **full**: every weight is trained. **partial**: only the top `PARTIAL_TOP_LAYERS` layers (+ final norm; for T5 the top decoder layers).
* **LoRA** (Hu et al. 2021): `h = W x + (alpha/r) B A x` on every linear layer, `W` frozen.
* **LoRMA** (compulsory): multiplicative low-rank update `h = (I + (alpha/r) B A) W x`.
* **Houlsby adapters** (Houlsby et al. 2019, Module 4.1): two bottleneck adapters per layer, `h + W_up ReLU(W_down h)`, applied to the output of self-attention and of the feed-forward block before the skip connection; near-identity start (`W_up = 0`); the layer norms are trained too.
* **Prefix tuning** (Li & Liang 2021, Module 4.1): `PREFIX_LENGTH` trainable key/value vectors are prepended at every layer (through the KV-cache input of the Phase-1 attention), reparametrized by an MLP for stable training; the whole Phase-1 model is frozen.
* **Pfeiffer adapters** (Pfeiffer et al. 2020, not in the lectures): one adapter after the feed-forward block only.
* **KD** (Hinton et al. 2015, word-level KD for generation as in Kim & Rush 2016): the student's next-token distribution on the summary tokens is pulled towards the teacher's softened distribution.

In [ ]:
import re
from safetensors.torch import load_file

def build_base(model_type, n_layers=None):
  V = len(bpe.itos)
  if model_type == "gpt":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="learned", norm_type="layernorm", ffn_type="gelu"))
  if model_type == "modern":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_KV_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="rope", norm_type="rmsnorm", ffn_type="swiglu"))
  if model_type == "rnn":
    return RNNLM(V, RNN_HIDDEN, n_layers or RNN_LAYERS, cell_type=RNN_CELL, dropout=DROPOUT)
  if model_type == "t5":
    return T5Model(V + T5_N_SENTINELS, bpe.pad_id, T5_D_MODEL, n_layers or T5_ENC_LAYERS, T5_DEC_LAYERS,
                   T5_HEADS, T5_D_FF, dropout=DROPOUT)
  raise ValueError(model_type)

def hidden_size(model_type):
  return {"gpt": N_EMBD, "modern": N_EMBD, "rnn": RNN_HIDDEN, "t5": T5_D_MODEL}[model_type]

def load_phase1_weights(base, model_type, n_layers=None):
  path = PHASE1_MODEL_FILES[model_type]
  state = load_file(path)
  if n_layers is not None:          # student: keep only the first n_layers layers
    kept = {}
    for k, v in state.items():
      m = re.match(r"^(blocks|cells|enc_layers)\.(\d+)\.", k)
      if m and int(m.group(2)) >= n_layers:
        continue
      kept[k] = v
    state = kept
  missing, unexpected = base.load_state_dict(state, strict=False)
  print(f"loaded Phase-1 {model_type} weights from {path} | missing {len(missing)} | unexpected {len(unexpected)}")
  if missing:
    raise ValueError(f"Phase-1 weights do not match the architecture, missing: {missing[:5]}")

def lm_hidden(base, input_ids):
  # DecoderLM.forward without the LM output layer
  B, T = input_ids.shape
  pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
  x = base.tok_emb(input_ids)
  if base.cfg.pos_emb_type == "learned":
    x = x + base.pos_emb(pos)
  x = base.drop(x)
  for block in base.blocks:
    x, _ = block(x, pos)
  return base.ln_f(x)

def rnn_hidden(base, input_ids):
  # RNNLM.forward without the LM output layer
  B, T = input_ids.shape
  x = base.dropout(base.tok_emb(input_ids))
  for l in range(base.num_layers):
    cell = base.cells[l]
    h = torch.zeros(B, base.hidden_size, device=x.device, dtype=x.dtype)
    state = (h, torch.zeros_like(h)) if base.is_lstm else h
    outputs = []
    for t in range(T):
      state = cell(x[:, t, :], state)
      outputs.append(state[0] if base.is_lstm else state)
    x = base.dropout(torch.stack(outputs, dim=1))
  return x


class LoRALinear(nn.Module):
  """LoRA: h = W x + b + scale * B A x  (W, b frozen); scale = alpha / r (LoRA) or alpha / sqrt(r) (rsLoRA)"""
  def __init__(self, base_linear, r, alpha, rank_stabilized=False):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.in_features) / math.sqrt(base_linear.in_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class RSLoRALinear(LoRALinear):
  def __init__(self, base_linear, r, alpha):
    super().__init__(base_linear, r, alpha, rank_stabilized=True)


class LoRMALinear(nn.Module):
  """LoRMA (additive identity): h = (I + (alpha / r) * B A) W x + b  (W, b frozen)"""
  def __init__(self, base_linear, r, alpha):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.out_features) / math.sqrt(base_linear.out_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    y = F.linear(x, self.base.weight)                      # W x
    y = y + self.scale * ((y @ self.A.t()) @ self.B.t())   # (I + s B A) W x
    if self.base.bias is not None:
      y = y + self.base.bias
    return y



def add_adapters(module, adapter_cls, r, alpha):
  # replace every nn.Linear inside the Phase-1 model by an adapter around it
  for name, child in module.named_children():
    if isinstance(child, nn.Linear):
      setattr(module, name, adapter_cls(child, r, alpha))
    else:
      add_adapters(child, adapter_cls, r, alpha)



class AdapterAfter(nn.Module):
  """Bottleneck adapter on the output of a sub-layer: h = sub(x);  h + W_up ReLU(W_down h).  W_up = 0 at the start."""
  def __init__(self, sub, d_model, bottleneck):
    super().__init__()
    self.sub = sub
    self.down = nn.Linear(d_model, bottleneck)
    self.up = nn.Linear(bottleneck, d_model)
    nn.init.zeros_(self.up.weight)
    nn.init.zeros_(self.up.bias)

  def forward(self, *args, **kwargs):
    out = self.sub(*args, **kwargs)
    if isinstance(out, tuple):                      # decoder attention returns (output, kv cache)
      h = out[0]
      return (h + self.up(F.relu(self.down(h))),) + tuple(out[1:])
    return out + self.up(F.relu(self.down(out)))


def add_bottleneck_adapters(base, model_type, d_model, after_attention):
  if model_type in ("gpt", "modern"):
    for block in base.blocks:
      block.ffn = AdapterAfter(block.ffn, d_model, BOTTLENECK)
      if after_attention:
        block.attn = AdapterAfter(block.attn, d_model, BOTTLENECK)
  elif model_type == "t5":
    for layer in list(base.enc_layers) + list(base.dec_layers):
      layer.ffn = AdapterAfter(layer.ffn, d_model, BOTTLENECK)
      if after_attention:
        layer.self_attn = AdapterAfter(layer.self_attn, d_model, BOTTLENECK)
  else:
    raise ValueError("bottleneck adapters are only defined for the Transformer models (gpt, modern, t5)")


def set_trainable(base, model_type, ft_method):
  for name, p in base.named_parameters():
    if ft_method == "full":
      p.requires_grad = True
    elif ft_method in ("lora", "lorma"):
      p.requires_grad = name.endswith(".A") or name.endswith(".B")
    elif ft_method in ("pfeiffer", "houlsby"):
      p.requires_grad = (".down." in name) or (".up." in name)
      if ft_method == "houlsby" and ("ln" in name or "norm" in name):
        p.requires_grad = True                  # Module 4.1: layer-norm parameters are trained per task
    else:
      p.requires_grad = False
  if ft_method == "partial":
    if model_type in ("gpt", "modern"):
      layers, final_norm = base.blocks, base.ln_f
    elif model_type == "rnn":
      layers, final_norm = base.cells, None
    else:
      layers, final_norm = base.dec_layers, base.dec_norm
    for layer in list(layers)[-PARTIAL_TOP_LAYERS:]:
      for p in layer.parameters():
        p.requires_grad = True
    if final_norm is not None:
      for p in final_norm.parameters():
        p.requires_grad = True


class PrefixTuning(nn.Module):
  """Prefix tuning for the decoder-only Phase-1 models: learned (key, value) prefixes of length P at every layer.
  P' (P x d) -> MLP -> keys and values of all layers (reparametrization), passed as past_key_values."""
  def __init__(self, base, n_prefix, hidden):
    super().__init__()
    self.base = base
    cfg = base.cfg
    self.P = n_prefix
    self.n_layers = cfg.n_layers
    self.n_kv = cfg.n_kv_heads
    self.d_head = cfg.d_model // cfg.n_heads
    self.prefix_emb = nn.Parameter(torch.randn(n_prefix, cfg.d_model) * 0.02)
    self.mlp = nn.Sequential(nn.Linear(cfg.d_model, hidden), nn.Tanh(),
                             nn.Linear(hidden, self.n_layers * 2 * self.n_kv * self.d_head))

  def prefix_kv(self, B):
    x = self.mlp(self.prefix_emb).view(self.P, self.n_layers, 2, self.n_kv, self.d_head)
    x = x.permute(1, 2, 3, 0, 4)                              # (layers, 2, kv heads, P, d_head)
    return [(x[l, 0].unsqueeze(0).expand(B, -1, -1, -1), x[l, 1].unsqueeze(0).expand(B, -1, -1, -1))
            for l in range(self.n_layers)]

  def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None, past_key_values=None, use_cache=False):
    B, T = input_ids.shape
    if past_key_values is None:                               # first call: start from the learned prefix
      past_key_values = self.prefix_kv(B)
      if position_ids is None:                                # real tokens keep positions 0, 1, 2, ... (as in Phase 1)
        position_ids = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
    past_len = past_key_values[0][0].size(2)
    if attention_mask is None:
      attention_mask = torch.ones(B, T, dtype=torch.long, device=input_ids.device)
    missing = past_len + T - attention_mask.size(1)           # the prefix positions are always visible
    if missing > 0:
      attention_mask = torch.cat([torch.ones(B, missing, dtype=attention_mask.dtype, device=input_ids.device), attention_mask], dim=1)
    return self.base(input_ids, labels=labels, attention_mask=attention_mask, position_ids=position_ids,
                     past_key_values=past_key_values, use_cache=use_cache)


def build_summ_model(model_type, ft_method, n_layers=None, load_pretrained=True):
  base = build_base(model_type, n_layers)
  if load_pretrained:
    load_phase1_weights(base, model_type, n_layers)
  if ft_method == "lora":
    add_adapters(base, LoRALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "lorma":
    add_adapters(base, LoRMALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method in ("pfeiffer", "houlsby"):
    add_bottleneck_adapters(base, model_type, hidden_size(model_type), after_attention=(ft_method == "houlsby"))
  set_trainable(base, model_type, ft_method)
  if ft_method == "prefix":
    if model_type not in ("gpt", "modern"):
      raise ValueError("prefix tuning is implemented for the decoder-only Transformers (gpt, modern)")
    return PrefixTuning(base, PREFIX_LENGTH, PREFIX_HIDDEN)   # base frozen, prefix + MLP trained
  return base


class KDSummModel(nn.Module):
  """word-level KD: alpha * CE(reference summary) + (1 - alpha) * T^2 * KL(teacher_T || student_T) on the summary tokens"""
  def __init__(self, student, teacher, temperature, alpha, seq2seq):
    super().__init__()
    self.student = student
    self.teacher = teacher
    self.T = temperature
    self.alpha = alpha
    self.seq2seq = seq2seq
    for p in self.teacher.parameters():
      p.requires_grad = False

  def train(self, mode=True):
    super().train(mode)
    self.teacher.eval()                     # the teacher never uses dropout
    return self

  def forward(self, input_ids, labels=None, decoder_input_ids=None):
    if self.seq2seq:
      out = self.student(input_ids, decoder_input_ids, labels)
    else:
      out = self.student(input_ids, labels=labels)
    if labels is None:
      return out
    with torch.no_grad():
      if self.seq2seq:
        t_logits = self.teacher(input_ids, decoder_input_ids)["logits"]
      else:
        t_logits = self.teacher(input_ids)["logits"]
    s_logits = out["logits"]
    if self.seq2seq:
      mask = labels != -100
    else:                                   # decoder-only: position t predicts token t+1
      mask = labels[:, 1:] != -100
      s_logits, t_logits = s_logits[:, :-1], t_logits[:, :-1]
    s = s_logits[mask].float() / self.T
    t = t_logits[mask].float() / self.T
    kd = F.kl_div(F.log_softmax(s, dim=-1), F.softmax(t, dim=-1), reduction="batchmean") * self.T * self.T
    return {"loss": self.alpha * out["loss"] + (1 - self.alpha) * kd, "logits": out["logits"]}


if USE_KD:
  assert (MODEL_TYPE == "t5") == (TEACHER_MODEL_TYPE == "t5"), "teacher and student must both be decoder-only, or both T5"
  student = build_summ_model(MODEL_TYPE, FT_METHOD, n_layers=STUDENT_LAYERS)
  teacher = build_summ_model(TEACHER_MODEL_TYPE, TEACHER_FT_METHOD, load_pretrained=False)
  teacher.load_state_dict(torch.load(TEACHER_SUMM_PATH, map_location="cpu"))
  print("teacher loaded from", TEACHER_SUMM_PATH)
  model = KDSummModel(student, teacher, KD_TEMPERATURE, KD_ALPHA, seq2seq=(MODEL_TYPE == "t5"))
  summ_model = student
  n_teacher = sum(p.numel() for p in teacher.parameters())
else:
  summ_model = build_summ_model(MODEL_TYPE, FT_METHOD, load_pretrained=USE_PHASE1_WEIGHTS)
  model = summ_model
  n_teacher = 0

n_total = sum(p.numel() for p in summ_model.parameters())
n_trainable = sum(p.numel() for p in summ_model.parameters() if p.requires_grad)
print(f"summarizer: {n_total / 1e6:.2f}M parameters | trainable {n_trainable / 1e6:.3f}M ({100 * n_trainable / n_total:.2f}%)")
if USE_KD:
  print(f"teacher: {n_teacher / 1e6:.2f}M parameters -> student is {100 * n_total / n_teacher:.0f}% of the teacher")

## 9. Training (Trainer, like the earlier phases)

The checkpoint is chosen by the validation loss on the summary tokens (generation at every epoch would be too slow). `prediction_loss_only` stops the Trainer from collecting the huge vocabulary logits.

In [ ]:
from transformers import TrainingArguments, Trainer

def make_args(out_name, epochs, with_eval):
  return TrainingArguments(
    output_dir=os.path.join(EXP_DIR, out_name),
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=5 if SMOKE_TEST else 50,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    fp16=torch.cuda.is_available() and MODEL_TYPE != "t5",   # T5 trains in full precision (fp16 broke it in Phase 2)
    eval_strategy="epoch" if with_eval else "no",
    save_strategy="epoch" if with_eval else "no",
    save_total_limit=1,
    load_best_model_at_end=with_eval,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,
    logging_steps=20,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
  )

if torch.cuda.is_available():
  torch.cuda.reset_peak_memory_stats()
start_time = time.time()
if LOAD_SUMM_PATH:
  summ_model.load_state_dict(torch.load(LOAD_SUMM_PATH, map_location="cpu"))
  print("loaded trained summarizer from", LOAD_SUMM_PATH, "- skipping training")
  log_history = []
else:
  if dapt_dataset is not None:
    print(f"domain adaptation: {DOMAIN_ADAPT_EPOCHS} epoch(s) of language modelling on the Phase-6 documents")
    dapt = Trainer(model=model, args=make_args("dapt", DOMAIN_ADAPT_EPOCHS, False), train_dataset=dapt_dataset,
                   data_collator=data_collator)
    dapt.train()
    print("   domain-adaptation loss:", [round(h["loss"], 3) for h in dapt.state.log_history if "loss" in h][-5:])
  trainer = Trainer(model=model, args=make_args("checkpoints", EPOCHS, True), train_dataset=train_dataset,
                    eval_dataset=val_dataset, data_collator=data_collator)
  trainer.train()
  log_history = trainer.state.log_history
training_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"Training time: {training_time / 60:.1f} min | peak GPU memory: {peak_memory_gb:.2f} GB")
for h in log_history:
  if "eval_loss" in h:
    print(f"epoch {h['epoch']:.2f}: validation loss {h['eval_loss']:.4f} (perplexity {math.exp(min(h['eval_loss'], 20)):.1f})")
  elif "loss" in h and int(h.get("step", 0)) % 100 == 0:
    print(f"   step {h['step']}: train loss {h['loss']:.4f}")

## 10. Generation (the Phase-1 loop: left padding, KV cache, `</s>` stop) + redundancy controls

* `NO_REPEAT_NGRAM`: a token that would repeat an earlier n-gram of the summary gets probability 0.
* `REPETITION_PENALTY` (Keskar et al. 2019): logits of already generated tokens are divided by the penalty.
* `MIN_NEW_TOKENS`: `</s>` is blocked at the start, so summaries are not cut too short.
* Post-processing: repeated sentences are removed; then the length control (`LENGTH_MODE`).

In [ ]:
def sample_next_token(logits, strategy="greedy", temperature=1.0, top_p=0.9):
  if strategy == "greedy":
    return torch.argmax(logits, dim=-1)
  probs = F.softmax(logits / temperature, dim=-1)
  sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
  keep = (torch.cumsum(sorted_probs, dim=-1) - sorted_probs) < top_p    # smallest set with mass >= top_p
  keep[:, 0] = True
  sampled_local = torch.multinomial(sorted_probs * keep, num_samples=1)
  return torch.gather(sorted_indices, -1, sampled_local).squeeze(1)

def left_pad(seqs, pad_id):
  L = max(len(s) for s in seqs)
  input_ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
  attention_mask = torch.zeros((len(seqs), L), dtype=torch.long)
  for i, s in enumerate(seqs):
    input_ids[i, L - len(s):] = torch.tensor(s, dtype=torch.long)
    attention_mask[i, L - len(s):] = 1
  return input_ids, attention_mask

def adjust_logits(logits, generated, ngram_tables, step):
  """repetition penalty, no-repeat n-gram and minimum length, in place. generated: list of token lists"""
  if step < MIN_NEW_TOKENS:
    logits[:, bpe.eos_id] = -1e9
  for b, seq in enumerate(generated):
    if REPETITION_PENALTY != 1.0 and seq:
      used = torch.tensor(sorted(set(seq)), device=logits.device)
      v = logits[b, used]
      logits[b, used] = torch.where(v > 0, v / REPETITION_PENALTY, v * REPETITION_PENALTY)
    if NO_REPEAT_NGRAM > 0 and len(seq) >= NO_REPEAT_NGRAM - 1:
      banned = ngram_tables[b].get(tuple(seq[len(seq) - (NO_REPEAT_NGRAM - 1):]), ())
      if banned:
        logits[b, list(banned)] = -1e9
  return logits

def remember_ngrams(generated, ngram_tables):
  if NO_REPEAT_NGRAM <= 0:
    return
  for b, seq in enumerate(generated):
    if len(seq) >= NO_REPEAT_NGRAM:
      key = tuple(seq[len(seq) - NO_REPEAT_NGRAM:len(seq) - 1])
      ngram_tables[b].setdefault(key, set()).add(seq[-1])

def remove_repeated_sentences(text):
  seen, out = set(), []
  for s in split_sentences(text):
    key = " ".join(content_words(s))
    if key and key in seen:
      continue
    seen.add(key)
    out.append(s)
  return " ".join(out)

def generate_batch(docs, model, model_type, max_new):
  device = next(model.parameters()).device
  V = len(bpe.itos)
  ctxs = [build_context(d, CONTEXT_MODE, DOC_BUDGET) for d in docs]
  B = len(docs)
  generated = [[] for _ in range(B)]
  ngram_tables = [dict() for _ in range(B)]
  finished = torch.zeros(B, dtype=torch.bool, device=device)
  with torch.inference_mode():
    if model_type == "t5":
      input_ids, _ = left_pad([T5_PREFIX_IDS + c for c in ctxs], bpe.pad_id)   # padding is masked in the encoder
      input_ids = input_ids.to(device)
      enc_out = model.encode(input_ids)
      dec_in = torch.full((B, 1), bpe.bos_id, dtype=torch.long, device=device)
      for step in range(max_new):
        logits = model.decode(dec_in, enc_out, input_ids, last_only=True)[:, -1, :V].float()
        logits = adjust_logits(logits, generated, ngram_tables, step)
        nxt = sample_next_token(logits, DECODING, TEMPERATURE, TOP_P)
        nxt = torch.where(finished, torch.full_like(nxt, bpe.pad_id), nxt)
        for b in range(B):
          if not finished[b]:
            generated[b].append(int(nxt[b]))
        remember_ngrams(generated, ngram_tables)
        finished = finished | (nxt == bpe.eos_id)
        if finished.all():
          break
        dec_in = torch.cat([dec_in, nxt.unsqueeze(1)], dim=1)
    else:
      prompts = [[bpe.bos_id] + c + SEP_IDS for c in ctxs]
      input_ids, attention_mask = left_pad(prompts, bpe.pad_id)
      input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
      if model_type == "rnn":
        out = model(input_ids, attention_mask=attention_mask, use_cache=True)
      else:
        position_ids = (attention_mask.cumsum(dim=-1) - 1).clamp(min=0)
        out = model(input_ids, attention_mask=attention_mask, position_ids=position_ids, use_cache=True)
        next_pos = position_ids[:, -1:] + 1
      past = out["past_key_values"]
      logits = out["logits"][:, -1, :].float()
      for step in range(max_new):
        logits = adjust_logits(logits, generated, ngram_tables, step)
        nxt = sample_next_token(logits, DECODING, TEMPERATURE, TOP_P)
        nxt = torch.where(finished, torch.full_like(nxt, bpe.pad_id), nxt)
        for b in range(B):
          if not finished[b]:
            generated[b].append(int(nxt[b]))
        remember_ngrams(generated, ngram_tables)
        finished = finished | (nxt == bpe.eos_id)
        if finished.all() or step == max_new - 1:
          break
        attention_mask = torch.cat([attention_mask, torch.ones((B, 1), dtype=attention_mask.dtype, device=device)], dim=1)
        if model_type == "rnn":
          out = model(nxt.unsqueeze(1), past_key_values=past, use_cache=True)
        else:
          out = model(nxt.unsqueeze(1), attention_mask=attention_mask, position_ids=next_pos,
                      past_key_values=past, use_cache=True)
          next_pos = next_pos + 1
        past = out["past_key_values"]
        logits = out["logits"][:, -1, :].float()
  texts = []
  for seq in generated:
    if bpe.eos_id in seq:
      seq = seq[:seq.index(bpe.eos_id)]
    seq = [t for t in seq if t < V]
    texts.append(remove_repeated_sentences(bpe.decode(seq)))
  return texts

def generate_all(docs, model, model_type):
  order = sorted(range(len(docs)), key=lambda i: docs[i]["n_tokens"])     # similar lengths in one batch
  out = [None] * len(docs)
  for s in range(0, len(order), GEN_BATCH_SIZE):
    idx = order[s:s + GEN_BATCH_SIZE]
    for i, t in zip(idx, generate_batch([docs[i] for i in idx], model, model_type, SUMMARY_TOKENS)):
      out[i] = t
  return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
summ_model = summ_model.to(device).eval()
gen_type = MODEL_TYPE
val_gen_docs = val_docs[:VAL_GEN_DOCS]
t0 = time.time()
val_generated = generate_all(val_gen_docs, summ_model, gen_type)
val_gen_seconds = time.time() - t0
print(f"generated {len(val_generated)} validation summaries in {val_gen_seconds:.0f}s ({val_gen_seconds / max(1, len(val_generated)):.1f}s each)")
for d, g in list(zip(val_gen_docs, val_generated))[:2]:
  print("\n--- validation", d["id"], "---")
  print("GENERATED:", g[:700])
  print("REFERENCE:", d["summary"][:700])

## 11. Validation scores, length control and analysis

* **ROUGE-1 / ROUGE-2 / ROUGE-L F1** (own code: word overlap and longest common subsequence, lower-cased words) and **BERTScore F1** (the primary metric; `bert_score` package, evaluation only).
* **Length:** each generated summary is also cut to several word lengths; the best one on validation is used for the test set (`LENGTH_MODE = "tuned"`). This shows how length moves BERTScore and ROUGE.
* **Baselines without a model:** the first N words of the judgment (lead), and the selected sentences (`"select"`), cut to the same length.
* **Redundancy:** share of repeated word 3-grams inside a summary. **Novelty (abstractiveness):** share of summary bigrams that are not in the judgment. **Facts:** recall of the reference's numbers (sections, dates, amounts) and the share of generated numbers that are not in the judgment (a hallucination signal).

In [ ]:
def words(text):
  return re.findall(r"[a-z0-9]+", text.lower())

def ngrams(w, n):
  return [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]

def f1_overlap(a, b):
  ca, cb = Counter(a), Counter(b)
  overlap = sum((ca & cb).values())
  if overlap == 0:
    return 0.0
  p, r = overlap / len(a), overlap / len(b)
  return 2 * p * r / (p + r)

def lcs_length(a, b):
  prev = [0] * (len(b) + 1)
  for x in a:
    cur = [0] * (len(b) + 1)
    for j, y in enumerate(b):
      cur[j + 1] = prev[j] + 1 if x == y else max(prev[j + 1], cur[j])
    prev = cur
  return prev[-1]

def rouge(pred, ref):
  p, r = words(pred), words(ref)
  if not p or not r:
    return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
  l = lcs_length(p, r)
  rl = 0.0 if l == 0 else 2 * (l / len(p)) * (l / len(r)) / (l / len(p) + l / len(r))
  return {"rouge1": f1_overlap(p, r), "rouge2": f1_overlap(ngrams(p, 2), ngrams(r, 2)), "rougeL": rl}

bert_scorer = None
if LOCAL_BERTSCORE:
  try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True)
    from bert_score import BERTScorer
    bert_scorer = BERTScorer(lang="en", device=str(device))
  except Exception as e:
    print("BERTScore not available:", e)

def score_set(preds, refs):
  rs = [rouge(p, r) for p, r in zip(preds, refs)]
  out = {k: float(np.mean([x[k] for x in rs])) for k in ["rouge1", "rouge2", "rougeL"]}
  if bert_scorer is not None:
    _, _, F1 = bert_scorer.score([p if p.strip() else "the" for p in preds], refs, batch_size=16)
    out["bertscore"] = float(F1.mean())
  return out

def cut_words(text, n):
  w = text.split()
  return text if n is None or len(w) <= n else " ".join(w[:n])

def show(name, sc):
  b = f" | BERTScore {sc['bertscore']:.4f}" if "bertscore" in sc else ""
  print(f"{name:34s} ROUGE-1 {sc['rouge1']:.4f} | ROUGE-2 {sc['rouge2']:.4f} | ROUGE-L {sc['rougeL']:.4f}{b}")

refs = [d["summary"] for d in val_gen_docs]
ref_len = [len(r.split()) for r in refs]
gen_len = [len(g.split()) for g in val_generated]
print(f"words per summary: reference median {int(np.median(ref_len))} | generated median {int(np.median(gen_len))} (max {max(gen_len)})")

sc_gen = score_set(val_generated, refs)
show("as generated", sc_gen)
train_sw = sorted(len(d["summary"].split()) for d in train_docs)
candidates = sorted(set(int(train_sw[int(q * (len(train_sw) - 1))]) for q in (0.25, 0.5, 0.75, 0.9)))
key = "bertscore" if bert_scorer is not None else "rouge1"
best_len, best_val = None, sc_gen[key]
length_curve = {"as generated": sc_gen}
for n in candidates:
  sc = score_set([cut_words(g, n) for g in val_generated], refs)
  length_curve[f"cut to {n} words"] = sc
  show(f"cut to {n} words", sc)
  if sc[key] > best_val:
    best_len, best_val = n, sc[key]
TUNED_WORDS = best_len
print(f"best length on validation ({key}): {'as generated' if TUNED_WORDS is None else str(TUNED_WORDS) + ' words'}")
final_val = [cut_words(g, TUNED_WORDS if LENGTH_MODE == "tuned" else None) for g in val_generated]
sc_final = score_set(final_val, refs)
show("used for the test set", sc_final)

L = TUNED_WORDS or int(np.median(ref_len))
sc_lead = score_set([" ".join(d["document"].split()[:L]) for d in val_gen_docs], refs)
show(f"baseline: first {L} words", sc_lead)
sc_sel = score_set([cut_words(bpe.decode(build_context(d, "select", DOC_BUDGET)), L) for d in val_gen_docs], refs)
show(f"baseline: selected sentences", sc_sel)

def repeated_trigram_share(text):
  g = ngrams(words(text), 3)
  return 1 - len(set(g)) / len(g) if g else 0.0

def novel_bigram_share(text, source):
  g = ngrams(words(text), 2)
  src = set(ngrams(words(source), 2))
  return sum(1 for x in g if x not in src) / len(g) if g else 0.0

def numbers(text):
  return set(re.findall(r"\b\d+[a-z]?\b", text.lower()))

redundancy = float(np.mean([repeated_trigram_share(g) for g in final_val]))
redundancy_ref = float(np.mean([repeated_trigram_share(r) for r in refs]))
novelty = float(np.mean([novel_bigram_share(g, d["document"]) for g, d in zip(final_val, val_gen_docs)]))
novelty_ref = float(np.mean([novel_bigram_share(r, d["document"]) for r, d in zip(refs, val_gen_docs)]))
num_recall = float(np.mean([len(numbers(g) & numbers(r)) / len(numbers(r)) for g, r in zip(final_val, refs) if numbers(r)] or [0]))
num_halluc = float(np.mean([len(numbers(g) - numbers(d["document"])) / len(numbers(g)) for g, d in zip(final_val, val_gen_docs) if numbers(g)] or [0]))
print(f"\nredundancy (repeated 3-grams): generated {100 * redundancy:.1f}% | reference {100 * redundancy_ref:.1f}%")
print(f"novel bigrams (abstractiveness): generated {100 * novelty:.1f}% | reference {100 * novelty_ref:.1f}%")
print(f"numbers (sections, dates, amounts): recall of the reference's numbers {100 * num_recall:.1f}% | generated numbers not in the judgment {100 * num_halluc:.1f}%")

thirds = np.array_split(np.argsort([d["n_tokens"] for d in val_gen_docs]), 3)
for name, idx in zip(["short", "medium", "long"], thirds):
  r1 = np.mean([rouge(final_val[i], refs[i])["rouge1"] for i in idx])
  print(f"{name:6s} documents ({len(idx)}): ROUGE-1 {r1:.4f} | input covers {100 * np.mean([min(1, DOC_BUDGET / val_gen_docs[i]['n_tokens']) for i in idx]):.0f}% of the document")

with open(os.path.join(EXP_DIR, "val_generated.json"), "w", encoding="utf-8") as f:
  json.dump([{"id": str(d["id"]), "generated": g, "reference": d["summary"]} for d, g in zip(val_gen_docs, final_val)], f, ensure_ascii=False)

## 12. Save the summarizer (needed when this experiment is the KD teacher)

In [ ]:
if LOAD_SUMM_PATH:
  print("generate-only run: model not saved again")
else:
  torch.save(summ_model.state_dict(), os.path.join(EXP_DIR, "summ_model.pt"))
  print("saved", os.path.join(EXP_DIR, "summ_model.pt"))

## 13. Test summaries -> prediction.json (list format) -> submission zip

Exactly one summary per test document: `[{"id": ..., "summary": ...}, ...]`.

In [ ]:
t0 = time.time()
test_generated = generate_all(test_docs, summ_model, gen_type)
test_seconds = time.time() - t0
predictions = []
for d, g in zip(test_docs, test_generated):
  s = cut_words(g, TUNED_WORDS if LENGTH_MODE == "tuned" else None).strip()
  if not s:                                   # never submit an empty summary
    s = " ".join(d["document"].split()[:L])
  predictions.append({"id": d["id"], "summary": s})
assert len(predictions) == len(test_docs) and len(set(str(p["id"]) for p in predictions)) == len(test_docs)
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
  json.dump(predictions, f, ensure_ascii=False)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
  z.write(PREDICTION_PATH, arcname="prediction.json")
n = [len(p["summary"].split()) for p in predictions]
print(f"{len(predictions)} test summaries in {test_seconds / 60:.1f} min | words: median {int(np.median(n))}, max {max(n)}")
print("submission:", SUBMISSION_ZIP)

## 14. Log this experiment (one line per experiment in results.jsonl)

In [ ]:
result = {
  "experiment": EXPERIMENT_NAME, "model": MODEL_TYPE, "method": FT_METHOD, "kd": USE_KD,
  "student_layers": STUDENT_LAYERS if USE_KD else None,
  "teacher": f"{TEACHER_MODEL_TYPE}/{TEACHER_FT_METHOD}" if USE_KD else None,
  "phase1_weights": USE_PHASE1_WEIGHTS, "domain_adapt_epochs": DOMAIN_ADAPT_EPOCHS,
  "train_docs": len(train_docs), "context_mode": CONTEXT_MODE, "doc_budget": DOC_BUDGET, "summary_tokens": SUMMARY_TOKENS,
  "input_coverage": round(cov[CONTEXT_MODE], 4),
  "decoding": DECODING, "no_repeat_ngram": NO_REPEAT_NGRAM, "repetition_penalty": REPETITION_PENALTY,
  "tuned_words": TUNED_WORDS, "length_mode": LENGTH_MODE,
  "params_M": round(n_total / 1e6, 2), "trainable_M": round(n_trainable / 1e6, 3),
  "trainable_pct": round(100 * n_trainable / n_total, 2),
  "train_min": round(training_time / 60, 1), "peak_mem_GB": round(peak_memory_gb, 2),
  "val_gen_s_per_doc": round(val_gen_seconds / max(1, len(val_generated)), 2), "test_gen_min": round(test_seconds / 60, 1),
  "best_val_loss": min([h["eval_loss"] for h in log_history if "eval_loss" in h] or [float("nan")]),
  "val_as_generated": {k: round(v, 4) for k, v in sc_gen.items()},
  "val_final": {k: round(v, 4) for k, v in sc_final.items()},
  "val_lead_baseline": {k: round(v, 4) for k, v in sc_lead.items()},
  "redundancy": round(redundancy, 4), "novel_bigrams": round(novelty, 4),
  "number_recall": round(num_recall, 4), "numbers_not_in_doc": round(num_halluc, 4),
  "lr": LEARNING_RATE, "epochs": EPOCHS,
}
with open(RESULTS_PATH, "a") as f:
  f.write(json.dumps(result) + "\n")
print(json.dumps(result, indent=1))